# Part 2: Text Classification on AG News Dataset 

### 2.1: Dataset: AG News

In [2]:
from datasets import load_dataset
import numpy as np

dataset = load_dataset("sh0416/ag_news")
print(dataset)

train_data = dataset['train']
test_data = dataset['test']
print(train_data[0])  
print(test_data[0])   

# Combine title + description into one text field
train_texts = [title + " " + desc for title, desc in zip(train_data['title'], train_data['description'])]
test_texts  = [title + " " + desc  for title, desc  in zip(test_data['title'],  test_data['description'])]


train_labels = np.array(train_data['label'])
test_labels  = np.array(test_data['label'])

print(f"Number of training samples: {len(train_texts)}")
print(f"Number of test samples: {len(test_texts)}")
print(f"Example training text (combined):\n{train_texts[0]}")
print(f"Example training label: {train_labels[0]}")
print(f"Unique labels: {np.unique(train_labels)}")           
print(f"Label mapping (typical): 1=World, 2=Sports, 3=Business, 4=Sci/Tech")

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 7600
    })
})
{'label': 3, 'title': 'Wall St. Bears Claw Back Into the Black (Reuters)', 'description': "Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again."}
{'label': 3, 'title': 'Fears for T N pension after talks', 'description': "Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul."}
Number of training samples: 120000
Number of test samples: 7600
Example training text (combined):
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
Example training label: 3
Unique labels: [1 2 3 4]
Label mapping (typical): 1=World, 2=Sports, 3=Business, 4=Sci/Tech


### 2.2: Preprocessing for Classification

In [3]:
import re
import string

def preprocess_text(text: str) -> list[str]:

    # Lowercase
    text = text.lower()
    
    #replace punctuation with space, for example: "can't" will be "can t"
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    
    #remove whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    #tokenization
    tokens = text.split()
    
    return tokens

example_text = train_texts[0]
tokens = preprocess_text(example_text)

print(f"Original text: {example_text}")
print(f"\nAfter preprocessing (tokens): {tokens}")  
print(f"\nNumber of tokens: {len(tokens)}")


print("Preprocessing all training texts...")
train_tokens_list = [preprocess_text(text) for text in train_texts]

print("Preprocessing all test texts...")
test_tokens_list = [preprocess_text(text) for text in test_texts]

print("Preprocessing finished!")
print(f"First training document tokens (first 10 words): {train_tokens_list[0][:10]}")
print(f"First test document tokens (first 10 words): {test_tokens_list[0][:10]}")

Original text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.

After preprocessing (tokens): ['wall', 'st', 'bears', 'claw', 'back', 'into', 'the', 'black', 'reuters', 'reuters', 'short', 'sellers', 'wall', 'street', 's', 'dwindling', 'band', 'of', 'ultra', 'cynics', 'are', 'seeing', 'green', 'again']

Number of tokens: 24
Preprocessing all training texts...
Preprocessing all test texts...
Preprocessing finished!
First training document tokens (first 10 words): ['wall', 'st', 'bears', 'claw', 'back', 'into', 'the', 'black', 'reuters', 'reuters']
First test document tokens (first 10 words): ['fears', 'for', 't', 'n', 'pension', 'after', 'talks', 'unions', 'representing', 'workers']


### 2.3: Naive Bayes Classification

**Class prior probability**

$$
P(c) = \frac{N_c}{N}
$$

Where:
- $N_c$ = number of documents in class $c$
- $N$ = total number of documents in the training set

**Likelihood (word probability given class) – with Laplace smoothing**

$$
P(w \mid c) = \frac{\text{count}(w, c) + \alpha}{\sum_{w'} \text{count}(w', c) + \alpha \cdot |V|}
$$

Where:
- $\text{count}(w, c)$ = number of times word $w$ appears in all documents of class $c$
- $\alpha$ = smoothing parameter (usually $\alpha = 1$ for Laplace smoothing)
- $|V|$ = vocabulary size (total number of unique words in the training set)
- $\sum_{w'} \text{count}(w', c)$ = total number of word tokens in class $c$

**Prediction – find the most probable class**

$$
\hat{c} = \arg\max_c \left( \log P(c) + \sum_{w \in d} \text{count}(w, d) \cdot \log P(w \mid c) \right)
$$

Where:
- $\hat{c}$ = predicted class for document $d$
- $\text{count}(w, d)$ = number of times word $w$ appears in document $d$
- The summation runs over all words that appear in the test document $d$
- Logarithms are used to avoid underflow and turn products into sums

#### 2.3.1: Implementation From Scratch

In [4]:
import numpy as np
from collections import Counter
from typing import List, Dict, Tuple

def train_multinomial_naive_bayes(
    train_tokens: List[List[str]],          
    train_labels: np.ndarray,               
    alpha: float = 1.0                      
) -> Dict:
    
    # Get classes and map them to 0-based indices
    classes = np.unique(train_labels)                   
    n_classes = len(classes)
    class_to_idx = {c: i for i, c in enumerate(classes)}

    n_docs = len(train_tokens)

    
    # Build vocabulary
    vocab_set = set()
    for tokens in train_tokens:
        vocab_set.update(tokens)
    
    vocab = sorted(list(vocab_set))
    vocab_size = len(vocab)
    word_to_idx = {w: i for i, w in enumerate(vocab)}

    print(f"Vocabulary size: {vocab_size:,}")


    # 2. Class priors: P(c) = (count(c) + 0) / N   
    class_counts = np.zeros(n_classes, dtype=np.float64)
    for lbl in train_labels:
        class_counts[class_to_idx[lbl]] += 1
    
    class_priors = class_counts / n_docs
    log_class_priors = np.log(class_priors + 1e-12)

    # 3. Word counts per class: count(w, c)
    count_matrix = np.zeros((n_classes, vocab_size), dtype=np.float64)
    
    for tokens, label in zip(train_tokens, train_labels):
        c_idx = class_to_idx[label]
        word_counter = Counter(tokens)
        for word, cnt in word_counter.items():
            if word in word_to_idx:                     
                widx = word_to_idx[word]
                count_matrix[c_idx, widx] += cnt

    # 4. Likelihood with Laplace smoothing
    #    P(w|c) = (count(w,c) + α) / (total_words_c + α * |V|)
    total_words_per_class = count_matrix.sum(axis=1)   
    total_words_per_class = np.maximum(total_words_per_class, 1e-10)
    
    likelihood = (count_matrix + alpha) / \
                 (total_words_per_class[:, np.newaxis] + alpha * vocab_size)
    
    log_likelihood = np.log(likelihood + 1e-12)

    # ───────────────────────────────────────────────
    # Package everything
    # ───────────────────────────────────────────────
    model = {
        'vocab': vocab,
        'word_to_idx': word_to_idx,
        'classes': classes,
        'log_class_priors': log_class_priors,           
        'log_likelihood': log_likelihood,               
        'alpha': alpha,
        'class_to_idx': class_to_idx
    }
    
    return model


def predict_naive_bayes(
    model: Dict,
    test_tokens_list: List[List[str]]
) -> np.ndarray:
    log_prior = model['log_class_priors']               
    log_like  = model['log_likelihood']                 
    vocab     = model['vocab']
    w2i       = model['word_to_idx']
    classes   = model['classes']
    
    n_classes = len(classes)
    predictions = []
    
    for tokens in test_tokens_list:
        # Score for each class: log P(c) + Σ log P(w|c) for w in document
        scores = np.full(n_classes, log_prior)          # start with log prior
        
        doc_counter = Counter(tokens)
        
        for word, count in doc_counter.items():
            if word in w2i:
                widx = w2i[word]
                # Add count * log P(w|c) for each class
                scores += count * log_like[:, widx]
        
        # Choose class with highest score
        best_idx = np.argmax(scores)
        pred_label = classes[best_idx]
        predictions.append(pred_label)
    
    return np.array(predictions)

In [5]:


print("Training Multinomial Naive Bayes...")
nb_model = train_multinomial_naive_bayes(
    train_tokens_list,
    train_labels,
    alpha=1.0
)

print("\nPredicting on test set...")
test_predictions = predict_naive_bayes(nb_model, test_tokens_list)

# Quick accuracy check (you can compare with test_labels)
accuracy = np.mean(test_predictions == test_labels)
print(f"Test accuracy (from-scratch NB):({accuracy*100:.2f}%)")

Training Multinomial Naive Bayes...
Vocabulary size: 65,015

Predicting on test set...
Test accuracy (from-scratch NB):(90.04%)


#### 2.3.2: Comparison with Scikit-learn

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline


# Create pipeline
nb_sklearn = Pipeline([
    ('vectorizer', CountVectorizer(
        lowercase=True,               # let it lowercase (matches our manual step)
        token_pattern=r'(?u)\b\w+\b', # simple word tokens
        stop_words=None,              # (matches our preprocessing)
        min_df=1                      # keep all terms
    )),
    ('classifier', MultinomialNB(alpha=1.0))   # same Laplace smoothing as from-scratch
])

print("Training scikit-learn pipeline on raw texts...")
nb_sklearn.fit(train_texts, train_labels)

print("Predicting on test set...")
sk_predictions = nb_sklearn.predict(test_texts)

#Evaluation 

sk_accuracy = accuracy_score(test_labels, sk_predictions)

print(f"\nscikit-learn MultinomialNB accuracy (raw texts): {sk_accuracy:.4f}  ({sk_accuracy*100:.2f}%)")
print(f"From-scratch NB accuracy (manual tokens):        (90.04%)")
print(f"Difference (sklearn raw - from-scratch):         {sk_accuracy - 0.9004:+.4f}")

# Detailed report
print("\nClassification Report (scikit-learn on raw texts):\n")
print(classification_report(test_labels, sk_predictions, 
                          target_names=['World', 'Sports', 'Business', 'Sci/Tech'],
                          digits=4))


Training scikit-learn pipeline on raw texts...
Predicting on test set...

scikit-learn MultinomialNB accuracy (raw texts): 0.9004  (90.04%)
From-scratch NB accuracy (manual tokens):        (90.04%)
Difference (sklearn raw - from-scratch):         -0.0000

Classification Report (scikit-learn on raw texts):

              precision    recall  f1-score   support

       World     0.9087    0.8958    0.9022      1900
      Sports     0.9490    0.9789    0.9637      1900
    Business     0.8786    0.8421    0.8600      1900
    Sci/Tech     0.8638    0.8847    0.8742      1900

    accuracy                         0.9004      7600
   macro avg     0.9000    0.9004    0.9000      7600
weighted avg     0.9000    0.9004    0.9000      7600



### 2.4: Logistic Regression

#### 2.4.1: Feature Representation (Word Bi-grams)
Each document is converted to binary features based on **word bi-grams** from the **training vocabulary**.
- Feature value = `1` if a bi-gram appears at least once in the document.
- Feature value = `0` otherwise.

In [7]:
import numpy as np
from typing import List, Tuple


def generate_bigrams(tokens: List[str]) -> List[str]:
    return [f"{tokens[i]}__{tokens[i+1]}" for i in range(len(tokens) - 1)]


def build_bigram_vocabulary(tokens_list: List[List[str]]) -> Tuple[List[str], dict, List[np.ndarray]]:
    # Build vocabulary from unique bigrams in training documents.
    vocab_set = set()
    train_doc_bigram_sets = []

    for tokens in tokens_list:
        doc_bigrams = set(generate_bigrams(tokens))
        train_doc_bigram_sets.append(doc_bigrams)
        vocab_set.update(doc_bigrams)

    vocab = sorted(vocab_set)
    bigram_to_idx = {bg: i for i, bg in enumerate(vocab)}

    doc_feature_indices = []
    for doc_bigrams in train_doc_bigram_sets:
        idxs = np.fromiter((bigram_to_idx[bg] for bg in doc_bigrams), dtype=np.int32)
        doc_feature_indices.append(idxs)

    return vocab, bigram_to_idx, doc_feature_indices


def vectorize_with_bigram_vocab(tokens_list: List[List[str]], bigram_to_idx: dict) -> List[np.ndarray]:
    doc_feature_indices = []
    for tokens in tokens_list:
        doc_bigrams = set(generate_bigrams(tokens))
        idxs = np.fromiter((bigram_to_idx[bg] for bg in doc_bigrams if bg in bigram_to_idx), dtype=np.int32)
        doc_feature_indices.append(idxs)
    return doc_feature_indices


print("Building word bi-gram vocabulary from training set...")
bigram_vocab, bigram_to_idx, X_train_indices = build_bigram_vocabulary(train_tokens_list)
X_test_indices = vectorize_with_bigram_vocab(test_tokens_list, bigram_to_idx)

print(f"Bi-gram vocabulary size: {len(bigram_vocab):,}")
print(f"First document active bi-grams: {len(X_train_indices[0])}")

# Map labels (1..4) to contiguous indices (0..3) for array indexing.
classes = np.sort(np.unique(train_labels))
label_to_idx = {lbl: i for i, lbl in enumerate(classes)}
y_train_idx = np.array([label_to_idx[lbl] for lbl in train_labels], dtype=np.int32)


# 2.4.2 Logistic Regression (multinomial / softmax) from scratch using NumPy

def softmax(logits: np.ndarray) -> np.ndarray:
    shifted = logits - np.max(logits)
    exp_vals = np.exp(shifted)
    return exp_vals / (np.sum(exp_vals) + 1e-12)


def train_softmax_logreg_sgd(
    X_indices: List[np.ndarray],
    y_idx: np.ndarray,
    num_features: int,
    num_classes: int,
    learning_rate: float = 0.1,
    epochs: int = 8,
    batch_size: int = 128,
    seed: int = 42
):
    rng = np.random.default_rng(seed)

    # Weight matrix shape: (num_classes, num_features)
    W = np.zeros((num_classes, num_features), dtype=np.float32)
    b = np.zeros(num_classes, dtype=np.float32)

    n_samples = len(X_indices)

    for epoch in range(1, epochs + 1):
        perm = rng.permutation(n_samples)  # shuffle each epoch
        epoch_loss = 0.0

        for start in range(0, n_samples, batch_size):
            batch_idx = perm[start:start + batch_size]

            grad_W = np.zeros_like(W)
            grad_b = np.zeros_like(b)
            batch_loss = 0.0

            for i in batch_idx:
                feat_idx = X_indices[i]
                logits = b.copy()
                if feat_idx.size > 0:
                    logits += W[:, feat_idx].sum(axis=1)

                probs = softmax(logits)

                yi = y_idx[i]
                batch_loss += -np.log(probs[yi] + 1e-12)

                error = probs.copy()
                error[yi] -= 1.0  # (p - y_one_hot)

                grad_b += error
                if feat_idx.size > 0:
                    grad_W[:, feat_idx] += error[:, np.newaxis]

            bs = len(batch_idx)
            grad_W /= bs
            grad_b /= bs
            batch_loss /= bs

            W -= learning_rate * grad_W
            b -= learning_rate * grad_b

            epoch_loss += batch_loss * bs

        epoch_loss /= n_samples
        print(f"Epoch {epoch:02d}/{epochs} - cross-entropy loss: {epoch_loss:.4f}")

    return W, b


def predict_softmax_logreg_idx(X_indices: List[np.ndarray], W: np.ndarray, b: np.ndarray) -> np.ndarray:
    preds = []
    for feat_idx in X_indices:
        logits = b.copy()
        if feat_idx.size > 0:
            logits += W[:, feat_idx].sum(axis=1)
        preds.append(np.argmax(logits))
    return np.array(preds, dtype=np.int32)


print("\nTraining from-scratch logistic regression (softmax + mini-batch SGD)...")
num_classes = len(classes)
num_features = len(bigram_vocab)

W_lr, b_lr = train_softmax_logreg_sgd(
    X_train_indices,
    y_train_idx,
    num_features=num_features,
    num_classes=num_classes,
    learning_rate=0.08,
    epochs=8,
    batch_size=128,
    seed=42
)

scratch_pred_idx = predict_softmax_logreg_idx(X_test_indices, W_lr, b_lr)
scratch_preds = classes[scratch_pred_idx]
scratch_acc = np.mean(scratch_preds == test_labels)
print(f"\nFrom-scratch Logistic Regression accuracy: {scratch_acc:.4f} ({scratch_acc*100:.2f}%)")

Building word bi-gram vocabulary from training set...
Bi-gram vocabulary size: 1,204,593
First document active bi-grams: 23

Training from-scratch logistic regression (softmax + mini-batch SGD)...
Epoch 01/8 - cross-entropy loss: 1.1351
Epoch 02/8 - cross-entropy loss: 0.8889
Epoch 03/8 - cross-entropy loss: 0.7724
Epoch 04/8 - cross-entropy loss: 0.6987
Epoch 05/8 - cross-entropy loss: 0.6458
Epoch 06/8 - cross-entropy loss: 0.6051
Epoch 07/8 - cross-entropy loss: 0.5722
Epoch 08/8 - cross-entropy loss: 0.5448

From-scratch Logistic Regression accuracy: 0.8424 (84.24%)


#### 2.4.3: Comparison with Scikit-learn

In [8]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, classification_report
from scipy.sparse import csr_matrix


def indices_to_csr(doc_indices: List[np.ndarray], vocab_size: int) -> csr_matrix:
    rows = []
    cols = []

    for r, idxs in enumerate(doc_indices):
        if idxs.size == 0:
            continue
        rows.extend([r] * len(idxs))
        cols.extend(idxs.tolist())

    data = np.ones(len(rows), dtype=np.float32)
    return csr_matrix((data, (rows, cols)), shape=(len(doc_indices), vocab_size), dtype=np.float32)


print("Converting bigram index lists to sparse binary matrices...")
X_train_csr = indices_to_csr(X_train_indices, len(bigram_vocab))
X_test_csr = indices_to_csr(X_test_indices, len(bigram_vocab))


print("\nTraining sklearn LogisticRegression...")
sk_lr = LogisticRegression(
    max_iter=200,
    solver='saga',
    random_state=42,
    n_jobs=-1
)
sk_lr.fit(X_train_csr, train_labels)
sk_lr_preds = sk_lr.predict(X_test_csr)
sk_lr_acc = accuracy_score(test_labels, sk_lr_preds)

print("Training sklearn SGDClassifier (logistic regression behavior)...")
sk_sgd = SGDClassifier(
    loss='log_loss',
    alpha=1e-5,
    max_iter=15,
    learning_rate='optimal',
    random_state=42
)
sk_sgd.fit(X_train_csr, train_labels)
sk_sgd_preds = sk_sgd.predict(X_test_csr)
sk_sgd_acc = accuracy_score(test_labels, sk_sgd_preds)

print("\n=== Logistic Regression Accuracy Comparison (Word Bi-grams, Binary Features) ===")
print(f"From-scratch (NumPy softmax SGD):      {scratch_acc:.4f} ({scratch_acc*100:.2f}%)")
print(f"sklearn LogisticRegression:            {sk_lr_acc:.4f} ({sk_lr_acc*100:.2f}%)")
print(f"sklearn SGDClassifier (log_loss):      {sk_sgd_acc:.4f} ({sk_sgd_acc*100:.2f}%)")

print("\nClassification report: sklearn LogisticRegression\n")
print(classification_report(
    test_labels,
    sk_lr_preds,
    target_names=['World', 'Sports', 'Business', 'Sci/Tech'],
    digits=4
))

Converting bigram index lists to sparse binary matrices...

Training sklearn LogisticRegression...
Training sklearn SGDClassifier (logistic regression behavior)...

=== Logistic Regression Accuracy Comparison (Word Bi-grams, Binary Features) ===
From-scratch (NumPy softmax SGD):      0.8424 (84.24%)
sklearn LogisticRegression:            0.9026 (90.26%)
sklearn SGDClassifier (log_loss):      0.9032 (90.32%)

Classification report: sklearn LogisticRegression

              precision    recall  f1-score   support

       World     0.9258    0.8932    0.9092      1900
      Sports     0.9369    0.9689    0.9527      1900
    Business     0.8816    0.8579    0.8696      1900
    Sci/Tech     0.8664    0.8905    0.8783      1900

    accuracy                         0.9026      7600
   macro avg     0.9027    0.9026    0.9024      7600
weighted avg     0.9027    0.9026    0.9024      7600



### 2.5: Confusion Matrix and Evaluation Metrics

#### 2.5.1: Implementation From Scratch (NumPy Only)

In [9]:
import numpy as np


def confusion_matrix_numpy(y_true: np.ndarray, y_pred: np.ndarray, labels: np.ndarray | None = None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    if labels is None:
        labels = np.sort(np.unique(np.concatenate([y_true, y_pred])))
    else:
        labels = np.asarray(labels)

    label_to_idx = {label: i for i, label in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=np.int64)

    for t, p in zip(y_true, y_pred):
        cm[label_to_idx[t], label_to_idx[p]] += 1

    return cm, labels


def classification_metrics_from_cm(cm: np.ndarray):
    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(axis=0).astype(np.float64) - tp
    fn = cm.sum(axis=1).astype(np.float64) - tp

    precision = tp / np.where(tp + fp == 0, 1.0, tp + fp)
    recall = tp / np.where(tp + fn == 0, 1.0, tp + fn)
    f1 = 2.0 * precision * recall / np.where(precision + recall == 0, 1.0, precision + recall)

    metrics = {
        'precision_per_class': precision,
        'recall_per_class': recall,
        'f1_per_class': f1,
        'precision_macro': float(np.mean(precision)),
        'recall_macro': float(np.mean(recall)),
        'f1_macro': float(np.mean(f1)),
        'accuracy': float(tp.sum() / max(cm.sum(), 1))
    }
    return metrics


def print_numpy_metrics(title: str, labels: np.ndarray, cm: np.ndarray, metrics: dict):
    print(f"\n{title}")
    print("Confusion Matrix (rows=true, cols=pred):")
    print(cm)

    print("\nPer-class metrics:")
    for i, lbl in enumerate(labels):
        print(
            f"Class {lbl}: "
            f"Precision={metrics['precision_per_class'][i]:.4f}, "
            f"Recall={metrics['recall_per_class'][i]:.4f}, "
            f"F1={metrics['f1_per_class'][i]:.4f}"
        )

    print("\nMacro-averaged:")
    print(f"Accuracy={metrics['accuracy']:.4f}")
    print(f"Precision_macro={metrics['precision_macro']:.4f}")
    print(f"Recall_macro={metrics['recall_macro']:.4f}")
    print(f"F1_macro={metrics['f1_macro']:.4f}")


variant_predictions = {
    'From-scratch Logistic Regression': scratch_preds,
    'sklearn LogisticRegression': sk_lr_preds,
    'sklearn SGDClassifier (log_loss)': sk_sgd_preds,
}

np_eval_results = {}
for model_name, preds in variant_predictions.items():
    cm_np, labels_eval = confusion_matrix_numpy(test_labels, preds, labels=classes)
    metrics_np = classification_metrics_from_cm(cm_np)
    np_eval_results[model_name] = {
        'cm': cm_np,
        'metrics': metrics_np,
        'labels': labels_eval,
    }
    print_numpy_metrics(f"NumPy Metrics - {model_name}", labels_eval, cm_np, metrics_np)


NumPy Metrics - From-scratch Logistic Regression
Confusion Matrix (rows=true, cols=pred):
[[1566  101   95  138]
 [  44 1756   29   71]
 [  84   61 1467  288]
 [  79   68  140 1613]]

Per-class metrics:
Class 1: Precision=0.8832, Recall=0.8242, F1=0.8527
Class 2: Precision=0.8842, Recall=0.9242, F1=0.9038
Class 3: Precision=0.8475, Recall=0.7721, F1=0.8080
Class 4: Precision=0.7645, Recall=0.8489, F1=0.8045

Macro-averaged:
Accuracy=0.8424
Precision_macro=0.8448
Recall_macro=0.8424
F1_macro=0.8422

NumPy Metrics - sklearn LogisticRegression
Confusion Matrix (rows=true, cols=pred):
[[1697   66   69   68]
 [  26 1841   17   16]
 [  64   29 1630  177]
 [  46   29  133 1692]]

Per-class metrics:
Class 1: Precision=0.9258, Recall=0.8932, F1=0.9092
Class 2: Precision=0.9369, Recall=0.9689, F1=0.9527
Class 3: Precision=0.8816, Recall=0.8579, F1=0.8696
Class 4: Precision=0.8664, Recall=0.8905, F1=0.8783

Macro-averaged:
Accuracy=0.9026
Precision_macro=0.9027
Recall_macro=0.9026
F1_macro=0.902

### 2.6: Comparison with Scikit-learn

In [10]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score


sk_eval_results = {}
for model_name, preds in variant_predictions.items():
    cm_sk = confusion_matrix(test_labels, preds, labels=classes)
    precision_sk_per = precision_score(test_labels, preds, labels=classes, average=None, zero_division=0)
    recall_sk_per = recall_score(test_labels, preds, labels=classes, average=None, zero_division=0)
    f1_sk_per = f1_score(test_labels, preds, labels=classes, average=None, zero_division=0)

    precision_sk_macro = precision_score(test_labels, preds, labels=classes, average='macro', zero_division=0)
    recall_sk_macro = recall_score(test_labels, preds, labels=classes, average='macro', zero_division=0)
    f1_sk_macro = f1_score(test_labels, preds, labels=classes, average='macro', zero_division=0)
    accuracy_sk = accuracy_score(test_labels, preds)

    sk_eval_results[model_name] = {
        'cm': cm_sk,
        'precision_per_class': precision_sk_per,
        'recall_per_class': recall_sk_per,
        'f1_per_class': f1_sk_per,
        'precision_macro': precision_sk_macro,
        'recall_macro': recall_sk_macro,
        'f1_macro': f1_sk_macro,
        'accuracy': accuracy_sk,
    }

    print(f"\nscikit-learn Metrics - {model_name}")
    print("Confusion Matrix (rows=true, cols=pred):")
    print(cm_sk)

    print("\nPer-class metrics:")
    for i, lbl in enumerate(classes):
        print(
            f"Class {lbl}: "
            f"Precision={precision_sk_per[i]:.4f}, "
            f"Recall={recall_sk_per[i]:.4f}, "
            f"F1={f1_sk_per[i]:.4f}"
        )

    print("\nMacro-averaged:")
    print(f"Accuracy={accuracy_sk:.4f}")
    print(f"Precision_macro={precision_sk_macro:.4f}")
    print(f"Recall_macro={recall_sk_macro:.4f}")
    print(f"F1_macro={f1_sk_macro:.4f}")


print("\nMatch checks (NumPy vs sklearn):")
for model_name in variant_predictions:
    np_res = np_eval_results[model_name]
    sk_res = sk_eval_results[model_name]
    print(f"\n{model_name}:")
    print(f"Confusion matrix equal:    {np.array_equal(np_res['cm'], sk_res['cm'])}")
    print(f"Precision per-class match: {np.allclose(np_res['metrics']['precision_per_class'], sk_res['precision_per_class'])}")
    print(f"Recall per-class match:    {np.allclose(np_res['metrics']['recall_per_class'], sk_res['recall_per_class'])}")
    print(f"F1 per-class match:        {np.allclose(np_res['metrics']['f1_per_class'], sk_res['f1_per_class'])}")
    print(f"Accuracy match:            {np.isclose(np_res['metrics']['accuracy'], sk_res['accuracy'])}")
    print(f"Precision macro match:     {np.isclose(np_res['metrics']['precision_macro'], sk_res['precision_macro'])}")
    print(f"Recall macro match:        {np.isclose(np_res['metrics']['recall_macro'], sk_res['recall_macro'])}")
    print(f"F1 macro match:            {np.isclose(np_res['metrics']['f1_macro'], sk_res['f1_macro'])}")


scikit-learn Metrics - From-scratch Logistic Regression
Confusion Matrix (rows=true, cols=pred):
[[1566  101   95  138]
 [  44 1756   29   71]
 [  84   61 1467  288]
 [  79   68  140 1613]]

Per-class metrics:
Class 1: Precision=0.8832, Recall=0.8242, F1=0.8527
Class 2: Precision=0.8842, Recall=0.9242, F1=0.9038
Class 3: Precision=0.8475, Recall=0.7721, F1=0.8080
Class 4: Precision=0.7645, Recall=0.8489, F1=0.8045

Macro-averaged:
Accuracy=0.8424
Precision_macro=0.8448
Recall_macro=0.8424
F1_macro=0.8422

scikit-learn Metrics - sklearn LogisticRegression
Confusion Matrix (rows=true, cols=pred):
[[1697   66   69   68]
 [  26 1841   17   16]
 [  64   29 1630  177]
 [  46   29  133 1692]]

Per-class metrics:
Class 1: Precision=0.9258, Recall=0.8932, F1=0.9092
Class 2: Precision=0.9369, Recall=0.9689, F1=0.9527
Class 3: Precision=0.8816, Recall=0.8579, F1=0.8696
Class 4: Precision=0.8664, Recall=0.8905, F1=0.8783

Macro-averaged:
Accuracy=0.9026
Precision_macro=0.9027
Recall_macro=0.9026
